# About the Dataset
1- id : unique id for a news article

2- title : the title of a news article

3- author : author of the news article

4- text : the text of the article; could be incomplete

5- label : a label that marked whether the news article is realor fake.

1 : Fake news

0 : real News

# Importing the Dependencies

In [79]:
pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [80]:
import numpy as np
import pandas as pd

import re #regular expression library for searching words in a text or paragraph
from nltk.corpus import stopwords #natural language toolkit / stopwords removes words that don't add much value (where, this, that, is, or, here...)
from nltk.stem.porter import PorterStemmer #take a word and removes the prefix and suffix then return the word (uses the root word of a word)

from sklearn.feature_extraction.text import TfidfVectorizer #convert words into vectors (numbers)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [81]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /home/chayma/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [82]:
print("the stop words that don't add value and should be removed in english : ",   stopwords.words('english'))

the stop words that don't add value and should be removed in english :  ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'our

## Data Preprocessing

In [83]:
1# loading the dataset to a pandas DataFrame

news_dataset = pd.read_csv('fake_news.csv')

In [84]:
2# check number of rows and columns

news_dataset.shape


(20800, 5)

In [85]:
# print the first 5 rows of the data

news_dataset.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [86]:
#counting the number of missing values in the dataset

news_dataset.isnull().sum()

id           0
title      558
author    1957
text        39
label        0
dtype: int64

In [87]:
# we have enough dataset to train the model so we can drop the missing values

# -> replacing the null values with empty string

news_dataset = news_dataset.fillna(' ')

In [88]:
# merging the authot name and news title (we won't use the tex)

news_dataset['content'] = news_dataset['author']+' '+news_dataset['title']

In [89]:
print(news_dataset['content'])

0        Darrell Lucus House Dem Aide: We Didn’t Even S...
1        Daniel J. Flynn FLYNN: Hillary Clinton, Big Wo...
2        Consortiumnews.com Why the Truth Might Get You...
3        Jessica Purkiss 15 Civilians Killed In Single ...
4        Howard Portnoy Iranian woman jailed for fictio...
                               ...                        
20795    Jerome Hudson Rapper T.I.: Trump a ’Poster Chi...
20796    Benjamin Hoffman N.F.L. Playoffs: Schedule, Ma...
20797    Michael J. de la Merced and Rachel Abrams Macy...
20798    Alex Ansary NATO, Russia To Hold Parallel Exer...
20799              David Swanson What Keeps the F-35 Alive
Name: content, Length: 20800, dtype: object


In [90]:
#separating the data and labels

X = news_dataset.drop(columns='label', axis=1)
Y = news_dataset['label']

print(X)
print(Y)

          id                                              title  \
0          0  House Dem Aide: We Didn’t Even See Comey’s Let...   
1          1  FLYNN: Hillary Clinton, Big Woman on Campus - ...   
2          2                  Why the Truth Might Get You Fired   
3          3  15 Civilians Killed In Single US Airstrike Hav...   
4          4  Iranian woman jailed for fictional unpublished...   
...      ...                                                ...   
20795  20795  Rapper T.I.: Trump a ’Poster Child For White S...   
20796  20796  N.F.L. Playoffs: Schedule, Matchups and Odds -...   
20797  20797  Macy’s Is Said to Receive Takeover Approach by...   
20798  20798  NATO, Russia To Hold Parallel Exercises In Bal...   
20799  20799                          What Keeps the F-35 Alive   

                                          author  \
0                                  Darrell Lucus   
1                                Daniel J. Flynn   
2                             Consortiu

## Stemming

Stemming : is the process of reducing a word to it's Root word

example : 
actor, actress. acting --> act

In [91]:
port_stem = PorterStemmer()

1- create a function called stemming and we give it the input 'content'

2- we call regular expression library thet is useful to look for a paragrapgh or a text

    - sub : subtitues certain values, ^ means exclosure. The function includes everything from a to z, and A to Z (we only want alphabets and words, not numbers or special characters)
    
    - ' ' : the numbers and special characters will be replaced by a space
    
3- we convert all of our text to lower case, otherwise the machine learning might see the upper case as a specific feature.    

4- we split our text and will be converted to a list

5- we take each word and we will stem it, but we will be removing the stop words so no need to stem them, but will be removed instead.

6- all words will be joined

7- return the function

In [92]:
def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]',' ', content)
    stemmed_content = stemmed_content.lower()
    stemmed_content = stemmed_content.split()
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in stopwords.words('english')]
    stemmed_content = ' '.join(stemmed_content)
    return stemmed_content

In [93]:
#applying the function

news_dataset['content'] = news_dataset['content'].apply(stemming)

In [94]:
print(news_dataset['content'])

0        darrel lucu hous dem aid even see comey letter...
1        daniel j flynn flynn hillari clinton big woman...
2                   consortiumnew com truth might get fire
3        jessica purkiss civilian kill singl us airstri...
4        howard portnoy iranian woman jail fiction unpu...
                               ...                        
20795    jerom hudson rapper trump poster child white s...
20796    benjamin hoffman n f l playoff schedul matchup...
20797    michael j de la merc rachel abram maci said re...
20798    alex ansari nato russia hold parallel exercis ...
20799                            david swanson keep f aliv
Name: content, Length: 20800, dtype: object


In [95]:
 #separating the data and label

X = news_dataset['content'].values
Y = news_dataset['label'].values

#the first one we did can be removed as this is the important one, the first one was just to show the labels and data

In [96]:
print(X)

['darrel lucu hous dem aid even see comey letter jason chaffetz tweet'
 'daniel j flynn flynn hillari clinton big woman campu breitbart'
 'consortiumnew com truth might get fire' ...
 'michael j de la merc rachel abram maci said receiv takeov approach hudson bay new york time'
 'alex ansari nato russia hold parallel exercis balkan'
 'david swanson keep f aliv']


In [97]:
print(Y)

[1 0 1 ... 0 1 1]


In [98]:
Y.shape

(20800,)

Computer cannot understant text, so we need to convert everything to meaningful number the computer can understand

tfidVectorizer :

tf: counts the number of times a particular word is repeated in a document, the repitition tells the model that it is an important word

idf: a word that is repeated several times might have no meaning or no weight to the model

feature vectors: change to numbers


In [100]:
# converting textual data to numerical data

vectorizer = TfidfVectorizer() #tf: term frequency and idf: inverse document frequency
vectorizer.fit(X)

X= vectorizer.transform(X)



In [101]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 210687 stored elements and shape (20800, 17128)>
  Coords	Values
  (0, 267)	0.2701012497770876
  (0, 2483)	0.36765196867972083
  (0, 2959)	0.24684501285337127
  (0, 3600)	0.3598939188262558
  (0, 3792)	0.27053324808454915
  (0, 4973)	0.23331696690935097
  (0, 7005)	0.2187416908935914
  (0, 7692)	0.24785219520671598
  (0, 8630)	0.2921251408704368
  (0, 8909)	0.36359638063260746
  (0, 13473)	0.2565896679337956
  (0, 15686)	0.2848506356272864
  (1, 1497)	0.2939891562094648
  (1, 1894)	0.15521974226349364
  (1, 2223)	0.3827320386859759
  (1, 2813)	0.19094574062359204
  (1, 3568)	0.26373768806048464
  (1, 5503)	0.7143299355715573
  (1, 6816)	0.1904660198296849
  (1, 16799)	0.30071745655510157
  (2, 2943)	0.3179886800654691
  (2, 3103)	0.46097489583229645
  (2, 5389)	0.3866530551182615
  (2, 5968)	0.3474613386728292
  (2, 9620)	0.49351492943649944
  :	:
  (20797, 3643)	0.2115550061362374
  (20797, 7042)	0.21799048897828685
  (2079

# Splitting the dataset to training and test data

In [108]:
X_train, X_test, Y_train, Y_test = train_test_split(X,Y,test_size=0.2, stratify=Y, random_state=2)

# Training the Model : Logistic Regression

In [110]:
model = LogisticRegression()

In [111]:
model.fit(X_train, Y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


## Evaluation

- Accuracy Score

In [112]:
# Accuracy score on training data

X_train_prediction = model.predict(X_train)
training_data_accuracy = accuracy_score(X_train_prediction, Y_train)

In [113]:
print('Accuracy score of the training data : ', training_data_accuracy)

Accuracy score of the training data :  0.9863581730769231


In [114]:
# Accuracy score on test data

X_test_prediction = model.predict(X_test)
test_data_accuracy = accuracy_score(X_test_prediction, Y_test)

In [115]:
print('Accuracy score of the test data : ', test_data_accuracy)

Accuracy score of the test data :  0.9790865384615385


# Predictive System

making a Predictive System

In [143]:
X_new = X_test[0]  #first row

prediction = model.predict(X_new)

In [144]:
print(prediction)

[1]


In [145]:
if (prediction[0] ==0):
    print('the news is Real')
else: 
    print('the news is fake')    

the news is fake


In [148]:
print(Y_test[0])

1
